# ISCO → O*NET Alternate Titles – Data Preview

Explore the crosswalk output before loading into the professions database.

**Source:** `scripts/output/isco_onet_alternate_titles.csv`  
**Columns:** `iscoGroup` · `onetSocCode` · `alternateTitle`

In [4]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("output/isco_onet_alternate_titles.csv")
df = pd.read_csv(CSV_PATH, dtype=str)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

Loaded 95,093 rows × 3 columns


## Preview Raw Data

In [2]:
df.head(15)

,iscoGroup,onetSocCode,alternateTitle
0,0110,55-1011.00,Airdrop Systems Technician
1,0110,55-1011.00,"Astronaut, Mission Specialist"
2,0110,55-1011.00,Helicopter Officer
3,0110,55-1011.00,"Naval Flight Officer, Airborne Reconnaissance ..."
4,0110,55-1011.00,"Naval Flight Officer, Bombardier/Navigator"
5,0110,55-1011.00,"Naval Flight Officer, Electronic Warfare Officer"
6,0110,55-1011.00,"Naval Flight Officer, Qualified Supporting Arm..."
7,0110,55-1011.00,"Naval Flight Officer, Radar Intercept Officer"
8,0110,55-1011.00,"Naval Flight Officer, Weapons Systems Officer"
9,0110,55-1011.00,Special Project Airborne Electronics Evaluator


In [3]:
df.sample(15, random_state=42)

,iscoGroup,onetSocCode,alternateTitle
12143,2310,25-1051.00,Paleontology Teacher
70472,8131,51-9195.00,Mold Builder
47617,6130,45-2021.00,Dog Breeder
2530,1213,11-9199.09,Wind Site Supervisor
75719,8157,51-6011.00,Table Machine Operator
86623,9312,47-3019.00,Post Tensioning Ironworker Helper
93100,9334,53-7065.00,Toolroom Attendant
66477,7549,51-9083.00,Contact Lens Blocker and Cutter
27373,3151,53-1043.00,On Car Supervisor
8768,2149,17-2131.00,Failure Analysis Engineer


## ISCO Preferred Title vs O*NET Alternate Titles

Compare the official ISCO-08 occupation title (closest to your `preferred_title`) with the O*NET alternate titles mapped to each ISCO group.

In [7]:
# Load ISCO-08 titles from the BLS crosswalk
isco_xwalk = pd.read_excel("data/ISCO_SOC_Crosswalk.xls", engine="xlrd", header=6)
isco_titles = (
    isco_xwalk[["ISCO-08 Code", "ISCO-08 Title EN"]]
    .drop_duplicates()
    .rename(columns={"ISCO-08 Code": "iscoGroup", "ISCO-08 Title EN": "preferredTitle"})
)
isco_titles["iscoGroup"] = isco_titles["iscoGroup"].astype(str).str.strip().str.zfill(4)
# Some codes are 3-digit (sub-major groups) — keep as-is after zfill
isco_titles = isco_titles[isco_titles["iscoGroup"].str.match(r"^\d{3,4}$")]

# Join with alternate titles
comparison = df.merge(isco_titles, on="iscoGroup", how="left")
print(f"{comparison['preferredTitle'].notna().sum():,} of {len(comparison):,} rows have an ISCO preferred title\n")

# Show side-by-side: preferred title vs alternate titles (grouped)
comparison[["iscoGroup", "preferredTitle", "onetSocCode", "alternateTitle"]].head(30)

95,093 of 95,093 rows have an ISCO preferred title



,iscoGroup,preferredTitle,onetSocCode,alternateTitle
0,0110,Commissioned armed forces officers,55-1011.00,Airdrop Systems Technician
1,0110,Commissioned armed forces officers,55-1011.00,"Astronaut, Mission Specialist"
2,0110,Commissioned armed forces officers,55-1011.00,Helicopter Officer
3,0110,Commissioned armed forces officers,55-1011.00,"Naval Flight Officer, Airborne Reconnaissance Officer"
4,0110,Commissioned armed forces officers,55-1011.00,"Naval Flight Officer, Bombardier/Navigator"
5,0110,Commissioned armed forces officers,55-1011.00,"Naval Flight Officer, Electronic Warfare Officer"
6,0110,Commissioned armed forces officers,55-1011.00,"Naval Flight Officer, Qualified Supporting Arms Coordinator (Airborne)"
7,0110,Commissioned armed forces officers,55-1011.00,"Naval Flight Officer, Radar Intercept Officer"
8,0110,Commissioned armed forces officers,55-1011.00,"Naval Flight Officer, Weapons Systems Officer"
9,0110,Commissioned armed forces officers,55-1011.00,Special Project Airborne Electronics Evaluator


In [8]:
# Compact view: one row per ISCO group showing the preferred title + all alternate titles
pd.set_option("display.max_colwidth", 120)

grouped = (
    comparison.groupby(["iscoGroup", "preferredTitle"])["alternateTitle"]
    .apply(lambda x: " | ".join(sorted(x.unique())[:15]))  # first 15 titles
    .reset_index()
    .rename(columns={"alternateTitle": "sampleAlternateTitles"})
)
grouped["altCount"] = comparison.groupby("iscoGroup")["alternateTitle"].count().values[:len(grouped)]

# Show the first 30 ISCO groups
grouped.head(30)

,iscoGroup,preferredTitle,sampleAlternateTitles,altCount
0,0110,Commissioned armed forces officers,AADC Plans Staff Officer | AOC AADC Chief of Operations Staff Officer | AOC AADC Director and Chief of Plans Staff O...,252
1,0210,Non-commissioned armed forces officers,"Aerial Gunner, Superintendent | Aerospace Control and Warning Systems Superintendent | Afloat Cryptologic Manager | ...",50
2,0211,Physical and earth science professionals,All Source Intelligence Analyst | Commercial Drone Operator | Commercial Drone Pilot | Drone Operator | Drone Pilot ...,51
3,0310,"Armed forces occupations, other ranks",ACDS Block 1 Operator | AEGIS Console Operator Track 3 | AN/SSN-2 (V) 4 Operator | AN/SYQ-13 NAV/C2 Operator | AN/TS...,462
4,0315,Ship and aircraft controllers and technicians,Airfield Operations Specialist | Airfield Services Officer | Airline Agent | Airline Dispatcher | Airport Agent | Ai...,28
5,1111,Legislators,Alderman | Assembly Member | Assembly Person | Assemblyman | Assemblywoman | City Alderman | City Council Member | C...,26
6,1112,Senior government officials,911 Communications Manager | Aeronautics Commission Director | Agency Owner | Agricultural Services Director | Area ...,197
7,1113,Traditional chiefs and heads of villages,Aeronautics Commission Director | Agency Owner | Agricultural Services Director | Alderman | Arts and Humanities Cou...,126
8,1114,Senior officials of special-interest organizations,Account Manager | Account Supervisor | Accreditation Lieutenant | Accreditation Manager | Activities Manager | Advan...,581
9,1120,Managing directors and chief executives,Aeronautics Commission Director | Agency Owner | Agricultural Services Director | Area Manager | Arts and Humanities...,174


In [9]:
# ✏️ Browse a specific group — change this code
BROWSE = "2512"

row = comparison[comparison["iscoGroup"] == BROWSE]
if len(row):
    title = row["preferredTitle"].iloc[0]
    alts = sorted(row["alternateTitle"].unique())
    print(f"ISCO {BROWSE}: {title}")
    print(f"{'─' * 60}")
    print(f"{len(alts)} alternate titles:\n")
    for i, t in enumerate(alts, 1):
        print(f"  {i:3d}. {t}")
else:
    print(f"No data for ISCO {BROWSE}")

ISCO 2512: Software developers
────────────────────────────────────────────────────────────
146 alternate titles:

    1. AI Specialist (Artificial Intelligence Specialist)
    2. Application Analyst
    3. Application Architect
    4. Application Developer
    5. Application Engineer
    6. Application Integration Engineer
    7. Application Integrator
    8. Application Programmer
    9. Application Software Engineering IT Specialist (Application Software Engineering Information Technology Specialist)
   10. Application Support Engineer
   11. Application Systems Analyst
   12. Application Systems Architect
   13. Applications Analyst
   14. Applications Quality Assurance Specialist (Applications QA Specialist)
   15. Applications Software Engineering Information Technology Specialist (Applications Software Engineering IT Specialist)
   16. Applications System Analyst
   17. Applications Tester
   18. Automation Specialist
   19. Automation Tester
   20. Beta Tester
   21. Bug Bounty

## Data Types, Shape & Missing Values

In [5]:
print(f"Shape: {df.shape}\n")
df.info()
print(f"\n=== Missing values ===")
print(df.isnull().sum())
print(f"\n=== Duplicate rows: {df.duplicated().sum()} ===")

Shape: (95093, 3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95093 entries, 0 to 95092
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   iscoGroup       95093 non-null  object
 1   onetSocCode     95093 non-null  object
 2   alternateTitle  95093 non-null  object
dtypes: object(3)
memory usage: 2.2+ MB

=== Missing values ===
iscoGroup         0
onetSocCode       0
alternateTitle    0
dtype: int64

=== Duplicate rows: 0 ===


## Summary Statistics

In [6]:
print(f"Unique ISCO groups:     {df['iscoGroup'].nunique()}")
print(f"Unique O*NET-SOC codes: {df['onetSocCode'].nunique()}")
print(f"Unique alternate titles: {df['alternateTitle'].nunique()}")
print()

# Title counts per ISCO group
group_counts = (
    df.groupby("iscoGroup")
    .agg(titles=("alternateTitle", "count"), onet_codes=("onetSocCode", "nunique"))
    .sort_values("titles", ascending=False)
)
group_counts.head(20)

Unique ISCO groups:     438
Unique O*NET-SOC codes: 1016
Unique alternate titles: 46687



,titles,onet_codes
iscoGroup,,
9329,4394,7
8142,1665,13
2310,1455,37
7223,1455,12
8181,1346,9
8141,1034,8
8114,1008,5
7543,940,1
3131,858,6


## Browse by ISCO Group

Change the `ISCO` variable below to explore titles for any group.  
Some common Proskiro codes: `2512` (software dev), `2511` (data analyst), `2411` (accountant), `2166` (graphic designer)

In [7]:
# ✏️ Change this to explore different ISCO groups
ISCO = "2512"  # Software and applications developers

subset = df[df["iscoGroup"] == ISCO].copy()
print(f"ISCO {ISCO}: {len(subset)} alternate titles across {subset['onetSocCode'].nunique()} O*NET codes\n")
subset

ISCO 2512: 161 alternate titles across 2 O*NET codes



,iscoGroup,onetSocCode,alternateTitle
16663,2512,15-1252.00,AI Specialist (Artificial Intelligence Special...
16664,2512,15-1252.00,Application Analyst
16665,2512,15-1252.00,Application Architect
16666,2512,15-1252.00,Application Developer
16667,2512,15-1252.00,Application Engineer
...,...,...,...
16819,2512,15-1253.00,User Experience Designer (UX Designer)
16820,2512,15-1253.00,User Interface Designer (UI Designer)
16821,2512,15-1253.00,User Interface and User Experience Architect (...
16822,2512,15-1253.00,User Interface and User Experience Designer (U...


## Cross-reference with Proskiro Profession Codes

Check how many alternate titles match your existing professions' ISCO groups.

In [8]:
import re

# Sample of profession codes from your DB
sample_proskiro_codes = [
    ("C2512.3",  "software developer"),
    ("C2511.2",  "data analyst"),
    ("C2511.3",  "data scientist"),
    ("C2421.1",  "business analyst"),
    ("C2166.10", "graphic designer"),
    ("C2411.1",  "accountant"),
    ("C2413.1",  "financial analyst"),
    ("C3314.1",  "actuarial assistant"),
    ("C2631.1",  "economist"),
    ("C2120.6",  "statistician"),
    ("C3313.2",  "bookkeeper"),
    ("C1346.1",  "bank manager"),
    ("C4313.1",  "payroll clerk"),
    ("C2513.5",  "web developer"),
    ("C2514.2",  "ICT application developer"),
]

def extract_isco(code: str) -> str:
    m = re.match(r"C(\d{4})", code)
    return m.group(1) if m else None

results = []
for code, title in sample_proskiro_codes:
    isco = extract_isco(code)
    if isco:
        count = len(df[df["iscoGroup"] == isco])
        sample_titles = df[df["iscoGroup"] == isco]["alternateTitle"].head(3).tolist()
        results.append({
            "proskiro_code": code,
            "profession": title,
            "isco_group": isco,
            "alt_titles_count": count,
            "sample_titles": " | ".join(sample_titles) if sample_titles else "—",
        })

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(results)

,proskiro_code,profession,isco_group,alt_titles_count,sample_titles
0,C2512.3,software developer,2512,161,AI Specialist (Artificial Intelligence Specialist) | Application Analyst | A...
1,C2511.2,data analyst,2511,147,Applications Analyst | Applications Systems Analyst | Automatic Data Process...
2,C2511.3,data scientist,2511,147,Applications Analyst | Applications Systems Analyst | Automatic Data Process...
3,C2421.1,business analyst,2421,121,Acquisition Analyst | Automated Logistics Specialist | Client Services Admin...
4,C2166.10,graphic designer,2166,105,3D Animator (Three-Dimensional Animator) | 3D Artist (Three-Dimensional Arti...
5,C2411.1,accountant,2411,110,Account Auditor | Accountant | Accounting Associate
6,C2413.1,financial analyst,2413,146,Commercial Credit Analyst | Commercial Credit Manager | Credit Administrator
7,C3314.1,actuarial assistant,3314,174,Analytics Consultant | Applied Scientist | Data Analyst
8,C2631.1,economist,2631,51,Agricultural Economist | Business Economist | Consultant Economist
9,C2120.6,statistician,2120,164,Actuarial Analyst | Actuarial Associate | Actuarial Consultant


## Title Length Analysis

Check for unusually long/short titles that might need cleaning before DB insertion.

In [ ]:
title_len = df["alternateTitle"].str.len()
print(title_len.describe())

print(f"\n--- Shortest titles (≤5 chars) ---")
short = df[title_len <= 5][["iscoGroup", "onetSocCode", "alternateTitle"]]
print(f"{len(short)} titles")
if len(short):
    print(short.to_string(index=False))

print(f"\n--- Longest titles (top 10) ---")
print(df.nlargest(10, df.columns[2], key=lambda s: s.str.len())[["iscoGroup", "onetSocCode", "alternateTitle"]].to_string(index=False))